This notebook serves a comparison purpose between few controll variables for future steps. The required setup is:

1. `density=0.25`
2. `shots=1000`
3. `seed=123` (following the previous IBM Fez experiment.)
4. `lengths = [2]+[4,10,20,50]`
5. Compare between `'edge_grab'`, `'matching'` and `'new'`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.transpiler import Target, CouplingMap
from qiskit.quantum_info import Operator
from qiskit.circuit.library import CXGate  # <-- Add this import
from qiskit_device_benchmarking.bench_code.mrb import MirrorQA, QuantumAwesomeness
import os, random

# Fix random seed for numpy
SEED = 123 # pick your favorite int
os.environ["PYTHONHASHSEED"] = str(SEED)  # optional, for hash-based determinism
random.seed(SEED)
np.random.seed(SEED)

In [2]:
# Define parameters for the simulated backend
num_qubits = 16
basis_gates = ["id", "h", "x", "y", "z", "rz", "cx"]
p2 = 1e-2 # 2-qubit gate error probability
p1 = p2/10  # 1-qubit gate error probability
rz_angle = np.pi / 2  # Match initial_entangling_angle

shots = 1000 # 10000
lengths = [2]+[4,10,20,50] # ,100
num_samples = 20
 
cmap = CouplingMap.from_grid(4, 4, bidirectional=True)

In [3]:
# Create a Target object to define the gates, including rz explicitly
target = Target.from_configuration(
    num_qubits=num_qubits,
    basis_gates=basis_gates,
    coupling_map=cmap,
    custom_name_mapping={
        "id": Operator(np.array([[1, 0], [0, 1]])).to_instruction(),  # Identity gate
        "h": Operator(np.array([[1, 1], [1, -1]]) / np.sqrt(2)).to_instruction(),  # Hadamard gate
        "x": Operator(np.array([[0, 1], [1, 0]])).to_instruction(),  # Pauli X gate
        "y": Operator(np.array([[0, -1j], [1j, 0]])).to_instruction(),  # Pauli Y gate
        "z": Operator(np.array([[1, 0], [0, -1]])).to_instruction(),  # Pauli Z gate
        "rz": Operator([[np.cos(rz_angle / 2), -1j*np.sin(rz_angle / 2)],
                        [-1j*np.sin(rz_angle / 2), np.cos(rz_angle / 2)]]).to_instruction(),  # RZ(π/2)
        # Use the built-in CXGate to ensure a valid Qiskit Instruction is used for two-qubit gates
        "cx": CXGate(),  # CNOT gate
    }
)

# Create a noise model to emulate the NoisyBackend
noise_model = NoiseModel()

# Add depolarizing errors for 1-qubit and 2-qubit gates
error_1q = depolarizing_error(p1, 1)
error_2q = depolarizing_error(p2, 2)

# Apply errors to all basis gates except 'delay' and 'reset'
for gate in basis_gates:
    if gate in ["id", "h", "x", "y", "z", "rz"]:
        noise_model.add_all_qubit_quantum_error(error_1q, gate)
    elif gate == "cx":
        noise_model.add_all_qubit_quantum_error(error_2q, gate)

In [4]:
# Set up the AerSimulator with stabilizer method, target, and noise model
backend = AerSimulator(
    method="stabilizer",
    noise_model=noise_model if (p1 > 0 or p2 > 0) else None,
    target=target,
    max_parallel_threads=0,
    max_parallel_experiments=0,
    seed_simulator=SEED
)

In [5]:
# Set up the experiment 01 ('edge_grab'+0.25) object
exp01 = MirrorQA(
    range(num_qubits),
    lengths=lengths,
    backend=backend,
    two_qubit_gate_density=0.25, # aim for 1.0 and 0.5 for mirror_qa_topo (0.25 from Fez)
    num_samples=num_samples,
    initial_entangling_angle=np.pi/2,
    sampling_algorithm='edge_grab', # 'edge_grab', 'matching', 'new'
    seed=SEED
)

# Set run options
exp01.set_run_options(shots=shots)

rb_data_01 = exp01.run()
print(rb_data_01.job_ids)

for i in range(len(exp01._pairs)):
    print(f"Pairs from round {i}", exp01._pairs[i])

['9a57150b-73f3-41ae-9c17-1196b7115792']
Pairs from round 0 [(0, 4), (1, 5), (12, 8)]
Pairs from round 1 [(2, 6), (4, 8), (3, 7)]
Pairs from round 2 [(10, 14), (8, 4), (13, 12), (15, 11), (2, 3)]
Pairs from round 3 [(1, 5), (7, 3), (4, 0), (13, 12)]
Pairs from round 4 [(4, 0), (15, 11), (3, 2), (13, 12)]
Pairs from round 5 [(9, 8), (1, 2), (7, 11), (14, 10), (12, 13)]
Pairs from round 6 [(3, 2), (11, 15), (0, 1), (6, 5)]
Pairs from round 7 [(13, 12), (0, 1), (11, 15)]
Pairs from round 8 [(9, 10), (3, 2), (5, 4)]
Pairs from round 9 [(4, 5), (7, 3), (1, 0)]
Pairs from round 10 [(15, 11), (6, 7)]
Pairs from round 11 [(4, 5), (9, 13), (6, 2), (14, 15), (3, 7)]
Pairs from round 12 [(10, 9), (7, 6), (15, 11), (13, 12)]
Pairs from round 13 [(11, 15), (14, 13)]
Pairs from round 14 [(10, 9), (7, 11), (6, 5), (12, 13)]
Pairs from round 15 [(10, 6), (2, 1), (4, 8), (11, 7), (14, 15)]
Pairs from round 16 [(6, 5), (15, 14), (13, 12), (11, 10), (9, 8), (3, 7)]
Pairs from round 17 [(8, 12), (2, 6)]
P

In [6]:
# Set up the experiment 02 ('matching'+0.25) object
exp02 = MirrorQA(
    range(num_qubits),
    lengths=lengths,
    backend=backend,
    two_qubit_gate_density=0.5, # aim for 1.0 and 0.5 for mirror_qa_topo (0.25 from Fez)
    num_samples=num_samples,
    initial_entangling_angle=np.pi/2,
    sampling_algorithm='matching', # 'edge_grab', 'matching', 'new'
    seed=SEED
)

# Set run options
exp02.set_run_options(shots=shots)

rb_data_02 = exp02.run()
print(rb_data_02.job_ids)

for i in range(len(exp02._pairs)):
    print(f"Pairs from round {i}", exp02._pairs[i])

['8adcde0a-ff4d-48f2-974a-8bb53e10602e']
Pairs from round 0 [(4, 0), (10, 11), (5, 1), (13, 9), (7, 6), (8, 12), (3, 2), (14, 15)]
Pairs from round 1 [(10, 11), (2, 3), (6, 7), (13, 9), (4, 5), (8, 12), (1, 0), (14, 15)]
Pairs from round 2 [(0, 1), (9, 13), (11, 10), (5, 4), (2, 3), (6, 7), (8, 12), (14, 15)]
Pairs from round 3 [(0, 1), (9, 13), (11, 10), (15, 14), (3, 7), (4, 5), (2, 6), (8, 12)]
Pairs from round 4 [(10, 11), (0, 4), (1, 5), (2, 3), (6, 7), (8, 9), (13, 12), (14, 15)]
Pairs from round 5 [(1, 2), (0, 4), (3, 7), (14, 13), (11, 15), (10, 6), (8, 12), (5, 9)]
Pairs from round 6 [(10, 11), (9, 13), (3, 7), (5, 4), (2, 6), (8, 12), (1, 0), (14, 15)]
Pairs from round 7 [(0, 1), (10, 11), (2, 3), (6, 7), (4, 5), (9, 8), (13, 12), (14, 15)]
Pairs from round 8 [(9, 10), (13, 14), (4, 0), (3, 7), (5, 1), (2, 6), (11, 15), (8, 12)]
Pairs from round 9 [(0, 1), (6, 2), (10, 14), (12, 13), (7, 3), (9, 5), (11, 15), (4, 8)]
Pairs from round 10 [(0, 1), (9, 10), (6, 2), (13, 14), (3,

In [7]:
lengths

[2, 4, 10, 20, 50]

In [8]:
# Set up the experiment 01 ('edge_grab'+0.25) object
exp03 = MirrorQA(
    range(num_qubits),
    lengths=lengths,
    backend=backend,
    two_qubit_gate_density=0.25, 
    num_samples=num_samples,
    initial_entangling_angle=np.pi/2,
    sampling_algorithm='new', # 'edge_grab', 'matching', 'new'
    seed=SEED
)

# Set run options
exp03.set_run_options(shots=shots)

rb_data_03 = exp03.run()
print(rb_data_03.job_ids)

# Print all available rounds, not a fixed range
for i in range(len(exp03._pairs)):
    print(f"Pairs from round {i}", exp03._pairs[i])

['98aee90c-2301-4a5e-ab46-19ded5e566b0']
Pairs from round 0 []
Pairs from round 1 [(4, 0), (10, 14), (1, 5), (12, 13), (2, 3), (6, 7), (8, 9), (11, 15)]
Pairs from round 2 []
Pairs from round 3 [(15, 11), (4, 0), (1, 2), (3, 7), (9, 5), (14, 13), (6, 10), (8, 12)]
Pairs from round 4 []
Pairs from round 5 []
Pairs from round 6 [(0, 1), (9, 13), (4, 5), (6, 10), (3, 2), (7, 11), (14, 15), (12, 8)]
Pairs from round 7 []
Pairs from round 8 [(0, 1), (6, 2), (10, 14), (3, 7), (5, 4), (12, 13), (8, 9), (11, 15)]
Pairs from round 9 []
Pairs from round 10 []
Pairs from round 11 [(0, 1), (15, 11), (12, 13), (14, 10), (2, 3), (6, 7), (4, 8), (5, 9)]
Pairs from round 12 []
Pairs from round 13 [(15, 14), (5, 4), (12, 13), (8, 9), (10, 6), (1, 0), (3, 2), (7, 11)]
Pairs from round 14 []
Pairs from round 15 []
Pairs from round 16 [(0, 1), (10, 14), (5, 4), (7, 3), (2, 6), (9, 8), (13, 12), (11, 15)]
Pairs from round 17 []
Pairs from round 18 [(6, 2), (0, 4), (1, 5), (7, 3), (14, 13), (11, 15), (8, 12